In [215]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

train_raw = pd.read_csv('train_data.csv')
test_raw = pd.read_csv('test_data.csv')

train_raw.head()

,ID,City A,City B,Distance,Time of Day,Weather,Traffic,Road Quality,Driver Experience,deliver_time
0,1,Satu Mare,Suceava,352,452,Fog,154.014691,370,30,355
1,2,Ploiesti,Timisoara,519,1386,Clear,949.697532,701,2,529
2,3,Deva,Bacau,457,91,Fog,387.019309,45,26,465
3,4,Hunedoara,Focsani,447,1120,Clear,130.544017,643,6,441
4,5,Hunedoara,Arad,201,1096,Clear,619.557737,375,20,230


In [224]:
# T1
task1ans = test_raw[(test_raw['City A'] == 'Barlad') & (test_raw['Weather'] == 'Fog')].shape[0]

In [218]:
# T2
colsToDrop = ['City A', 'City B', 'ID']

x_train = train_raw.drop(columns=colsToDrop + ['deliver_time'])

y_train = train_raw['deliver_time']

x_test = test_raw.drop(columns=colsToDrop)

oneHotCols = ['Weather']
numCols = ['Distance', 'Time of Day', 'Traffic', 'Road Quality', 'Driver Experience']

preprocessor = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore'), oneHotCols),
    ('scale', StandardScaler(), numCols)
])

pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', XGBRegressor(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 300],
    'model__max_depth': [3, 6],
    'model__learning_rate': [0.05, 0.1],
    'model__subsample': [0.8, 1.0],
}

gridSearch = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    verbose=1,
    n_jobs=-1
)
gridSearch.fit(x_train, y_train)
model = gridSearch.best_estimator_
predictions = model.predict(x_test)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


In [225]:
# Dataframe
rows = []
rows.append({
    'subtaskID': 1,
    'datapointID': 1,
    'answer': task1ans
})
for id,pred in zip(test_raw['ID'], predictions):
    rows.append({
        'subtaskID': 2,
        'datapointID': id,
        'answer': round(pred)
})
subDf = pd.DataFrame(rows)
subDf.to_csv('submission.csv', index=False)